# Part B: How the class designed WWI's Star Schema

The Week 7 PDF walks through a 4-step dimensional modeling process using WideWorldImporters.
This notebook follows the same steps — but instead of just reading slides, we query the actual database to verify.

## Setup — Connect to SQL Server

In [23]:
import pyodbc
import pandas as pd

conn = pyodbc.connect(
    r"DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=localhost;"
    r"DATABASE=WideWorldImporters;"
    r"Trusted_Connection=yes;"
    r"MARS_Connection=Yes;"
)

def sql(query):
    """Run a query and return results as a DataFrame."""
    return pd.read_sql(query, conn)

print("Connected to WideWorldImporters")

Connected to WideWorldImporters


## Step 1 — Find the Fact: what business event are we tracking?

![p.26](images/week7-p26.png)

> "We would like to track sales information for WideWorldImporters."

The "sales" = the business event = our Fact table's source.
First question: **where does sales data live in this database?**

In [24]:
# What schemas exist?
# Schema = namespace that groups tables, like folders for files.
# e.g. Sales.Orders means the Orders table inside the Sales schema.
sql("SELECT SCHEMA_NAME FROM INFORMATION_SCHEMA.SCHEMATA ORDER BY SCHEMA_NAME")

C:\Users\blitz\AppData\Local\Temp\ipykernel_17896\243355505.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,SCHEMA_NAME
0,Application
1,DataLoadSimulation
2,db_accessadmin
3,db_backupoperator
4,db_datareader
5,db_datawriter
6,db_ddladmin
7,db_denydatareader
8,db_denydatawriter
9,db_owner


Schemas we care about:
- **Sales** — the requirement says "track sales information", so this is our first target
- **Purchasing** — Suppliers live here (DimSuppliers source, needed for the assignment)
- **Warehouse** — StockItems, Colors (DimProducts source)
- **Application** — People, Cities, StateProvinces, Countries (DimSalesPeople, DimLocation source)

The rest (`db_*`, `sys`, `INFORMATION_SCHEMA`, etc.) are system internals — ignore.

Let's start with Sales since that's what the requirement asks us to track.

In [25]:
# The requirement says "track sales information"
# We saw a "Sales" schema in the schema list above — that's our starting point.
# Let's see what tables are inside it.
sql("""
    SELECT TABLE_NAME 
    FROM INFORMATION_SCHEMA.TABLES 
    WHERE TABLE_SCHEMA = 'Sales' 
    ORDER BY TABLE_NAME
""")

C:\Users\blitz\AppData\Local\Temp\ipykernel_17896\243355505.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,TABLE_NAME
0,BuyingGroups
1,BuyingGroups_Archive
2,CustomerCategories
3,CustomerCategories_Archive
4,Customers
5,Customers_Archive
6,CustomerTransactions
7,InvoiceLines
8,Invoices
9,OrderLines


The requirement says "track sales information" — so Sales schema is our starting point.
We're looking for the core of the business process: **what happens when a sale occurs?**

Which tables here are relevant?
- `Orders` + `OrderLines` → the core — an order (who, when) and its line items (what, how much). This is the business event we want to track, so we start here.
- `Customers` + `CustomerCategories` → customer info. Orders has `CustomerID` which points here — we'll follow this ID later to build a Dimension.
- `_Archive` tables → system temporal tables (SQL Server tracks history automatically), not our concern
- `Invoices`, `BuyingGroups`, `SpecialDeals` → exist in Sales but not part of this assignment's scope

**Approach:** Start with Orders/OrderLines (the business event) → discover what IDs they reference → follow those IDs to other tables to find the Dimensions.

In [26]:
# What does an actual order look like?
sql("SELECT TOP 5 * FROM Sales.Orders")

C:\Users\blitz\AppData\Local\Temp\ipykernel_17896\243355505.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,OrderID,CustomerID,SalespersonPersonID,PickedByPersonID,ContactPersonID,BackorderOrderID,OrderDate,ExpectedDeliveryDate,CustomerPurchaseOrderNumber,IsUndersupplyBackordered,Comments,DeliveryInstructions,InternalComments,PickingCompletedWhen,LastEditedBy,LastEditedWhen
0,1,832,2,NaN,3032,45.0,2013-01-01,2013-01-02,12126,True,None,None,None,2013-01-01 12:00:00,7,2013-01-01 12:00:00
1,2,803,8,NaN,3003,46.0,2013-01-01,2013-01-02,15342,True,None,None,None,2013-01-01 12:00:00,7,2013-01-01 12:00:00
2,3,105,7,NaN,1209,47.0,2013-01-01,2013-01-02,12211,True,None,None,None,2013-01-01 12:00:00,7,2013-01-01 12:00:00
3,4,57,16,3.0,1113,NaN,2013-01-01,2013-01-02,17129,True,None,None,None,2013-01-01 11:00:00,3,2013-01-01 11:00:00
4,5,905,3,NaN,3105,48.0,2013-01-01,2013-01-02,10369,True,None,None,None,2013-01-01 12:00:00,7,2013-01-01 12:00:00


Notice: `CustomerID = 832`, `SalespersonPersonID = 2` — just ID numbers, no names, no details.
This is 3NF — data is split across separate tables to avoid redundancy. To find out *who* customer 832 is or *where* they're located, you'd need to JOIN to other tables. We'll explore those in Step 3.

Also notice: **Orders has no measures** (no Quantity, no Price). Those live in OrderLines (next cell).

In [27]:
# What does an order LINE look like? (individual items within an order)
sql("SELECT TOP 5 * FROM Sales.OrderLines")

C:\Users\blitz\AppData\Local\Temp\ipykernel_17896\243355505.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,OrderLineID,OrderID,StockItemID,Description,PackageTypeID,Quantity,UnitPrice,TaxRate,PickedQuantity,PickingCompletedWhen,LastEditedBy,LastEditedWhen
0,1,45,164,32 mm Double sided bubble wrap 50m,7,50,112.0,15.0,50,2013-01-02 11:00:00,4,2013-01-02 11:00:00
1,2,1,67,Ride on toy sedan car (Black) 1/12 scale,7,10,230.0,15.0,10,2013-01-01 11:00:00,3,2013-01-01 11:00:00
2,3,2,50,Developer joke mug - old C developers never di...,7,9,13.0,15.0,9,2013-01-01 11:00:00,3,2013-01-01 11:00:00
3,4,46,89,"""The Gu"" red shirt XML tag t-shirt (Black) 3XS",7,72,18.0,15.0,72,2013-01-02 11:00:00,4,2013-01-02 11:00:00
4,5,46,171,32 mm Anti static bubble wrap (Blue) 10m,7,90,32.0,15.0,90,2013-01-02 11:00:00,4,2013-01-02 11:00:00


**Observations:**
- `Quantity`, `UnitPrice`, `TaxRate` — numeric data exists here (not in Orders)
- `OrderID = 46` appears in rows 3 and 4 — one order can have multiple lines
- Same OrderID, but different `StockItemID` — **StockItemID is what creates the fine grain within each order**
- No `CustomerID` or `SalespersonID` here — those were in Orders, not OrderLines

**To verify later:**
- Are Quantity/UnitPrice/TaxRate the measures for our Fact table? → Step 4
- What does StockItemID point to? → Step 3 (DimProducts)
- How do we combine Orders (who, when) + OrderLines (what, how much)? → need a JOIN

**How Orders and OrderLines relate:**
```
Orders (1 row = 1 order)
┌─────────┬────────────┬─────────────────────┬────────────┐
│ OrderID │ CustomerID │ SalespersonPersonID │ OrderDate  │
├─────────┼────────────┼─────────────────────┼────────────┤
│ 1       │ 832        │ 2                   │ 2013-01-01 │
└─────────┴────────────┴─────────────────────┴────────────┘
    │
    │ 1 order → many lines (FK: OrderLines.OrderID → Orders.OrderID)
    ▼
OrderLines (1 row = 1 product in that order)
┌─────────┬─────────────┬──────────┬───────────┬─────────┐
│ OrderID │ StockItemID │ Quantity │ UnitPrice │ TaxRate │
├─────────┼─────────────┼──────────┼───────────┼─────────┤
│ 1       │ 67          │ 10       │ 230.00    │ 15.0    │
│ 1       │ 50          │ 9        │ 13.00     │ 15.0    │
│ 1       │ ...         │ ...      │ ...       │ ...     │
└─────────┴─────────────┴──────────┴───────────┴─────────┘

Orders has WHO and WHEN.
OrderLines has WHAT and HOW MUCH.
They connect through OrderID.
```

In [28]:
# How big is the data we're working with?
# If OrderLines >> Orders, it confirms one order has multiple lines.
sql("""
    SELECT 
        (SELECT COUNT(*) FROM Sales.Orders) AS TotalOrders,
        (SELECT COUNT(*) FROM Sales.OrderLines) AS TotalOrderLines
""")

C:\Users\blitz\AppData\Local\Temp\ipykernel_17896\243355505.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,TotalOrders,TotalOrderLines
0,73595,231412


**Step 1 Summary:**
- Orders + OrderLines = **FactSales의 소스** (the business event we're tracking)
- Orders (73,595) = order headers — has CustomerID, SalespersonID, OrderDate (who, when)
- OrderLines (231,412) = line items — has StockItemID, Quantity, UnitPrice, TaxRate (what, how much)
- The IDs in Orders/OrderLines (CustomerID, SalespersonPersonID, StockItemID) point to other tables — those become Dimensions

## Step 2 — Decide the Fact's granularity

![p.27](images/week7-p27.png)

> "An order for one or more of a stock item by a customer in a city via a sales person."

We found the Fact source (Orders + OrderLines). Now: **should each row in FactSales be 1 order or 1 order line?**

![p.27](images/week7-p27.png)

> "An order for one or more of a stock item by a customer in a city via a sales person."

We found the Fact source (Orders + OrderLines). Now: **should each row in FactSales be 1 order or 1 order line?**

In [56]:
# Option A: OrderLine level (fine grain) — each product is its own row
sql("""
    SELECT o.OrderID, o.CustomerID, o.SalespersonPersonID, o.OrderDate,
           ol.StockItemID, ol.Quantity, ol.UnitPrice
    FROM Sales.Orders o
    JOIN Sales.OrderLines ol ON o.OrderID = ol.OrderID
    WHERE o.OrderID = 1
""")

C:\Users\blitz\AppData\Local\Temp\ipykernel_17896\243355505.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,OrderID,CustomerID,SalespersonPersonID,OrderDate,StockItemID,Quantity,UnitPrice
0,1,832,2,2013-01-01,67,10,230.0


In [55]:
# Option B: Order level (coarse grain) — products merged into totals
sql("""
    SELECT o.OrderID, o.CustomerID, o.OrderDate,
           SUM(ol.Quantity) AS TotalQuantity,
           SUM(ol.Quantity * ol.UnitPrice) AS TotalAmount
    FROM Sales.Orders o
    JOIN Sales.OrderLines ol ON o.OrderID = ol.OrderID
    WHERE o.OrderID = 1
    GROUP BY o.OrderID, o.CustomerID, o.OrderDate
""")

C:\Users\blitz\AppData\Local\Temp\ipykernel_17896\243355505.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,OrderID,CustomerID,OrderDate,TotalQuantity,TotalAmount
0,1,832,2013-01-01,10,2300.0


**Compare the two:**

The business requirement says:
> "analyze which **products** are being ordered to determine how **brand, colour, and price** may impact gross sales"

Option A (fine grain) — each product is its own row:
- "Which product sold the most?" → answerable ✅
- "Does brand affect sales?" → answerable ✅ (StockItemID links to product details)
- "Total order amount?" → SUM and you get it ✅ (roll-up always works)

Option B (coarse grain) — products merged into 1 row:
- "Total order amount?" → answerable ✅
- "Which product sold the most?" → **impossible** ❌ (StockItemID is gone)
- "Does brand affect sales?" → **impossible** ❌

The business needs product-level analysis → we need StockItemID in each row → **OrderLine level (Option A)**.

A → B is just a GROUP BY (always possible).
B → A is impossible — the detail is lost.

## Step 3 — Define Dimensions: follow the IDs

![p.28](images/week7-p28.png)
![p.29](images/week7-p29.png)

The class extracted nouns from the requirements and filtered:
- brand, colour → properties of product (not separate dimensions)
- price → a measure, not a dimension
- time = date (synonymous)
- city → could be customer property, but location is a common dimension → kept as DimCities

**Result: 5 Dimensions** — product, customer, city (location), salesperson, date

In Step 1, we saw that Orders/OrderLines only have IDs:
- Orders: `CustomerID`, `SalespersonPersonID`, `OrderDate`
- OrderLines: `StockItemID`

Now we follow those IDs to find what tables they point to and what data they hold.
Each ID trail becomes a Dimension.

First — we saw IDs in Orders and OrderLines, but where do they actually point?
Let's check the FK relationships before assuming anything.

In [57]:
# Orders FK — where do CustomerID, SalespersonPersonID actually point?
sql("""
    SELECT 
        COL_NAME(fkc.parent_object_id, fkc.parent_column_id) AS OrdersColumn,
        OBJECT_SCHEMA_NAME(fkc.referenced_object_id) + '.' + 
            OBJECT_NAME(fkc.referenced_object_id) AS ReferencedTable,
        COL_NAME(fkc.referenced_object_id, fkc.referenced_column_id) AS ReferencedColumn
    FROM sys.foreign_keys fk
    JOIN sys.foreign_key_columns fkc ON fk.object_id = fkc.constraint_object_id
    WHERE fk.parent_object_id = OBJECT_ID('Sales.Orders')
    ORDER BY OrdersColumn
""")

C:\Users\blitz\AppData\Local\Temp\ipykernel_17896\243355505.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,OrdersColumn,ReferencedTable,ReferencedColumn
0,BackorderOrderID,Sales.Orders,OrderID
1,ContactPersonID,Application.People,PersonID
2,CustomerID,Sales.Customers,CustomerID
3,LastEditedBy,Application.People,PersonID
4,PickedByPersonID,Application.People,PersonID
5,SalespersonPersonID,Application.People,PersonID


In [58]:
# OrderLines FK — where does StockItemID actually point?
sql("""
    SELECT 
        COL_NAME(fkc.parent_object_id, fkc.parent_column_id) AS OrderLinesColumn,
        OBJECT_SCHEMA_NAME(fkc.referenced_object_id) + '.' + 
            OBJECT_NAME(fkc.referenced_object_id) AS ReferencedTable,
        COL_NAME(fkc.referenced_object_id, fkc.referenced_column_id) AS ReferencedColumn
    FROM sys.foreign_keys fk
    JOIN sys.foreign_key_columns fkc ON fk.object_id = fkc.constraint_object_id
    WHERE fk.parent_object_id = OBJECT_ID('Sales.OrderLines')
    ORDER BY OrderLinesColumn
""")

C:\Users\blitz\AppData\Local\Temp\ipykernel_17896\243355505.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,OrderLinesColumn,ReferencedTable,ReferencedColumn
0,LastEditedBy,Application.People,PersonID
1,OrderID,Sales.Orders,OrderID
2,PackageTypeID,Warehouse.PackageTypes,PackageTypeID
3,StockItemID,Warehouse.StockItems,StockItemID


**Confirmed FK targets:**

From Orders:
- `CustomerID` → **Sales.Customers** (CustomerID)
- `SalespersonPersonID` → **Application.People** (PersonID)

From OrderLines:
- `StockItemID` → **Warehouse.StockItems** (StockItemID)
- `OrderID` → **Sales.Orders** (OrderID) — this is how OrderLines connects back to Orders

The rest (`BackorderOrderID`, `ContactPersonID`, `PickedByPersonID`, `PackageTypeID`, `LastEditedBy`) are not needed for our Dimensions.

Now let's follow each confirmed FK to build our Dimensions.

### DimCustomers

Orders has `CustomerID`. Let's follow it.

#### 1. CustomerID → Sales.Customers columns

In [45]:
# What columns does Customers have?
sql("""
    SELECT COLUMN_NAME, DATA_TYPE, IS_NULLABLE
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = 'Sales' AND TABLE_NAME = 'Customers'
    ORDER BY ORDINAL_POSITION
""")

C:\Users\blitz\AppData\Local\Temp\ipykernel_17896\243355505.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,COLUMN_NAME,DATA_TYPE,IS_NULLABLE
0,CustomerID,int,NO
1,CustomerName,nvarchar,NO
2,BillToCustomerID,int,NO
3,CustomerCategoryID,int,NO
4,BuyingGroupID,int,YES
5,PrimaryContactPersonID,int,NO
6,AlternateContactPersonID,int,YES
7,DeliveryMethodID,int,NO
8,DeliveryCityID,int,NO
9,PostalCityID,int,NO


**Observations:**
- `CustomerName` — finally a name, not just an ID. This goes into DimCustomers.
- `CustomerCategoryID`, `DeliveryCityID`, `PostalCityID` — more IDs pointing somewhere. But where?
- 30 columns total, but not all are useful for sales analysis

**Which columns matter for analyzing sales?**
- Useful: `CustomerName`, `CustomerCategoryID` (→ category name), `DeliveryCityID` / `PostalCityID` (→ city info)
- Not useful for this analysis: `CreditLimit`, `PaymentDays`, `DeliveryAddressLine1` — operational details, not analytical dimensions
- `PostalCode` could be useful in practice (zone-level analysis within a city), but the class scope stays at City level. In real projects, this would be decided with BA/SME.

**Next:** Before following those IDs, let's check where they actually point to — what tables are they FK'd to?

In [46]:
# Where do the FK columns in Customers actually point to?
sql("""
    SELECT 
        COL_NAME(fkc.parent_object_id, fkc.parent_column_id) AS CustomerColumn,
        OBJECT_SCHEMA_NAME(fkc.referenced_object_id) + '.' + 
            OBJECT_NAME(fkc.referenced_object_id) AS ReferencedTable,
        COL_NAME(fkc.referenced_object_id, fkc.referenced_column_id) AS ReferencedColumn
    FROM sys.foreign_keys fk
    JOIN sys.foreign_key_columns fkc ON fk.object_id = fkc.constraint_object_id
    WHERE fk.parent_object_id = OBJECT_ID('Sales.Customers')
    ORDER BY CustomerColumn
""")

C:\Users\blitz\AppData\Local\Temp\ipykernel_17896\243355505.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,CustomerColumn,ReferencedTable,ReferencedColumn
0,AlternateContactPersonID,Application.People,PersonID
1,BillToCustomerID,Sales.Customers,CustomerID
2,BuyingGroupID,Sales.BuyingGroups,BuyingGroupID
3,CustomerCategoryID,Sales.CustomerCategories,CustomerCategoryID
4,DeliveryCityID,Application.Cities,CityID
5,DeliveryMethodID,Application.DeliveryMethods,DeliveryMethodID
6,LastEditedBy,Application.People,PersonID
7,PostalCityID,Application.Cities,CityID
8,PrimaryContactPersonID,Application.People,PersonID


#### 2. Customers FK relationships

In [47]:
# CustomerCategoryID → what's in CustomerCategories?
sql("SELECT * FROM Sales.CustomerCategories")

C:\Users\blitz\AppData\Local\Temp\ipykernel_17896\243355505.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,CustomerCategoryID,CustomerCategoryName,LastEditedBy,ValidFrom,ValidTo
0,1,Agent,1,2013-01-01 00:00:00,9999-12-31 23:59:59.999999
1,2,Wholesaler,1,2013-01-01 00:00:00,9999-12-31 23:59:59.999999
2,3,Novelty Shop,1,2013-01-01 00:00:00,9999-12-31 23:59:59.999999
3,4,Supermarket,1,2013-01-01 00:00:00,9999-12-31 23:59:59.999999
4,5,Computer Store,1,2013-01-01 00:00:00,9999-12-31 23:59:59.999999
5,6,Gift Store,1,2013-01-01 00:00:00,9999-12-31 23:59:59.999999
6,7,Corporate,1,2013-01-01 00:00:00,9999-12-31 23:59:59.999999
7,8,General Retailer,9,2014-01-01 16:15:00,9999-12-31 23:59:59.999999


#### 3. Follow the FKs — CustomerCategories, Cities chain

In [48]:
# DeliveryCityID → what's in Cities?
# (avoiding SELECT * because geography column causes pyodbc error)
sql("""
    SELECT TOP 5 CityID, CityName, StateProvinceID
    FROM Application.Cities
""")

C:\Users\blitz\AppData\Local\Temp\ipykernel_17896\243355505.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,CityID,CityName,StateProvinceID
0,1,Aaronsburg,39
1,3,Abanda,1
2,4,Abbeville,42
3,5,Abbeville,11
4,6,Abbeville,1


Cities has `CityName` (the actual name we want) and `StateProvinceID` — another FK. Where does it point?

In [49]:
# Cities FK → where does StateProvinceID point?
sql("""
    SELECT 
        COL_NAME(fkc.parent_object_id, fkc.parent_column_id) AS CityColumn,
        OBJECT_SCHEMA_NAME(fkc.referenced_object_id) + '.' + 
            OBJECT_NAME(fkc.referenced_object_id) AS ReferencedTable,
        COL_NAME(fkc.referenced_object_id, fkc.referenced_column_id) AS ReferencedColumn
    FROM sys.foreign_keys fk
    JOIN sys.foreign_key_columns fkc ON fk.object_id = fkc.constraint_object_id
    WHERE fk.parent_object_id = OBJECT_ID('Application.Cities')
    ORDER BY CityColumn
""")

C:\Users\blitz\AppData\Local\Temp\ipykernel_17896\243355505.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,CityColumn,ReferencedTable,ReferencedColumn
0,LastEditedBy,Application.People,PersonID
1,StateProvinceID,Application.StateProvinces,StateProvinceID


`StateProvinceID` → **Application.StateProvinces** confirmed. Next link in the chain:

In [50]:
# And StateProvinces FK → where does CountryID point?
sql("""
    SELECT 
        COL_NAME(fkc.parent_object_id, fkc.parent_column_id) AS StateProvColumn,
        OBJECT_SCHEMA_NAME(fkc.referenced_object_id) + '.' + 
            OBJECT_NAME(fkc.referenced_object_id) AS ReferencedTable,
        COL_NAME(fkc.referenced_object_id, fkc.referenced_column_id) AS ReferencedColumn
    FROM sys.foreign_keys fk
    JOIN sys.foreign_key_columns fkc ON fk.object_id = fkc.constraint_object_id
    WHERE fk.parent_object_id = OBJECT_ID('Application.StateProvinces')
    ORDER BY StateProvColumn
""")

C:\Users\blitz\AppData\Local\Temp\ipykernel_17896\243355505.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,StateProvColumn,ReferencedTable,ReferencedColumn
0,CountryID,Application.Countries,CountryID
1,LastEditedBy,Application.People,PersonID


`CountryID` → **Application.Countries** confirmed. Is Countries the end of the chain?

In [51]:
# Countries FK → does it link to anything else?
sql("""
    SELECT 
        COL_NAME(fkc.parent_object_id, fkc.parent_column_id) AS CountryColumn,
        OBJECT_SCHEMA_NAME(fkc.referenced_object_id) + '.' + 
            OBJECT_NAME(fkc.referenced_object_id) AS ReferencedTable,
        COL_NAME(fkc.referenced_object_id, fkc.referenced_column_id) AS ReferencedColumn
    FROM sys.foreign_keys fk
    JOIN sys.foreign_key_columns fkc ON fk.object_id = fkc.constraint_object_id
    WHERE fk.parent_object_id = OBJECT_ID('Application.Countries')
    ORDER BY CountryColumn
""")

C:\Users\blitz\AppData\Local\Temp\ipykernel_17896\243355505.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,CountryColumn,ReferencedTable,ReferencedColumn
0,LastEditedBy,Application.People,PersonID


Only `LastEditedBy` (system column) — no more business FKs. **Countries is the end of the chain.**

Full location chain confirmed via FK:
```
Customers.DeliveryCityID
  → Application.Cities (CityName)
    → Application.StateProvinces (StateProvinceCode, StateProvinceName)
      → Application.Countries (CountryName, FormalName)
```
This entire chain will be flattened into DimLocation and DimCustomers.

#### 4. Denormalized result — what DimCustomers will look like

In [52]:
# Customers denormalized — JOIN to categories + delivery/postal cities
sql("""
    SELECT TOP 5
        cu.CustomerID, cu.CustomerName,
        cc.CustomerCategoryName,
        dc.CityName AS DeliveryCityName,
        dsp.StateProvinceCode AS DeliveryStateProvCode,
        dco.CountryName AS DeliveryCountryName,
        pc.CityName AS PostalCityName,
        psp.StateProvinceCode AS PostalStateProvCode,
        pco.CountryName AS PostalCountryName
    FROM Sales.Customers cu
    JOIN Sales.CustomerCategories cc ON cu.CustomerCategoryID = cc.CustomerCategoryID
    JOIN Application.Cities dc ON cu.DeliveryCityID = dc.CityID
    JOIN Application.StateProvinces dsp ON dc.StateProvinceID = dsp.StateProvinceID
    JOIN Application.Countries dco ON dsp.CountryID = dco.CountryID
    JOIN Application.Cities pc ON cu.PostalCityID = pc.CityID
    JOIN Application.StateProvinces psp ON pc.StateProvinceID = psp.StateProvinceID
    JOIN Application.Countries pco ON psp.CountryID = pco.CountryID
""")

C:\Users\blitz\AppData\Local\Temp\ipykernel_17896\243355505.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,CustomerID,CustomerName,CustomerCategoryName,DeliveryCityName,DeliveryStateProvCode,DeliveryCountryName,PostalCityName,PostalStateProvCode,PostalCountryName
0,1,Tailspin Toys (Head Office),Novelty Shop,Lisco,NE,United States,Lisco,NE,United States
1,2,"Tailspin Toys (Sylvanite, MT)",Novelty Shop,Sylvanite,MT,United States,Sylvanite,MT,United States
2,3,"Tailspin Toys (Peeples Valley, AZ)",Novelty Shop,Peeples Valley,AZ,United States,Peeples Valley,AZ,United States
3,4,"Tailspin Toys (Medicine Lodge, KS)",Novelty Shop,Medicine Lodge,KS,United States,Medicine Lodge,KS,United States
4,5,"Tailspin Toys (Gasport, NY)",Novelty Shop,Gasport,NY,United States,Gasport,NY,United States


**This is what denormalization looks like.** 8 tables JOINed into 1 flat row:
- `CustomerCategoryID = 3` → now we see "Novelty Shop"
- `DeliveryCityID = ?` → now we see "Impact, TX, United States"

No more IDs, no more JOINs needed.

Notice: Delivery and Postal are often identical — but they can differ, so we keep both.

**Should DimCustomers include location?**
Logically, no — FactSales already has `LocationKey` → DimLocation, which tracks where each order happened.
But the class SQL (Week 9 PDF p.18) includes location in DimCustomers. We follow the class design.
This means DimCustomers has redundant location data, but it's intentional — the class chose this structure.

#### 5. DimCustomers without location — if location belongs in DimLocation

In [67]:
# DimCustomers — strictly what the requirement asks for (no location)
sql("""
    SELECT TOP 5
        cu.CustomerID, cu.CustomerName,
        cc.CustomerCategoryName
    FROM Sales.Customers cu
    JOIN Sales.CustomerCategories cc ON cu.CustomerCategoryID = cc.CustomerCategoryID
""")

C:\Users\blitz\AppData\Local\Temp\ipykernel_17896\243355505.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,CustomerID,CustomerName,CustomerCategoryName
0,1,Tailspin Toys (Head Office),Novelty Shop
1,2,"Tailspin Toys (Sylvanite, MT)",Novelty Shop
2,3,"Tailspin Toys (Peeples Valley, AZ)",Novelty Shop
3,4,"Tailspin Toys (Medicine Lodge, KS)",Novelty Shop
4,5,"Tailspin Toys (Gasport, NY)",Novelty Shop


**Compare:** The full denormalized version above has 8 columns (with location). This version has 3 columns.
The class chose the 8-column version, but strictly by requirement, location belongs in DimLocation.
We follow the class design for this assignment.

### DimSalesPeople

Orders FK confirmed: `SalespersonPersonID` → **Application.People** (PersonID).
But People is a general table — not everyone is a salesperson. Let's explore.

#### 1. SalespersonPersonID → Application.People columns

In [59]:
# What columns does People have?
sql("""
    SELECT COLUMN_NAME, DATA_TYPE, IS_NULLABLE
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = 'Application' AND TABLE_NAME = 'People'
    ORDER BY ORDINAL_POSITION
""")

C:\Users\blitz\AppData\Local\Temp\ipykernel_17896\243355505.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,COLUMN_NAME,DATA_TYPE,IS_NULLABLE
0,PersonID,int,NO
1,FullName,nvarchar,NO
2,PreferredName,nvarchar,NO
3,SearchName,nvarchar,NO
4,IsPermittedToLogon,bit,NO
5,LogonName,nvarchar,YES
6,IsExternalLogonProvider,bit,NO
7,HashedPassword,varbinary,YES
8,IsSystemUser,bit,NO
9,IsEmployee,bit,NO


**Observations:**
- This is a general People table — not just salespeople. Has `IsSalesperson`, `IsEmployee`, `IsSystemUser` flags.
- Useful for DimSalesPeople: `FullName`, `PreferredName`, `LogonName`, `PhoneNumber`, `FaxNumber`, `EmailAddress`
- Not useful: `HashedPassword`, `Photo`, `CustomFields`, `UserPreferences` — system/operational data
- `IsSalesperson = 1` is the filter to get only salespeople out of this table

**Does People have FKs to other tables we need to follow?**

#### 2. People FK relationships

In [60]:
# People FK — any chains to follow?
sql("""
    SELECT 
        COL_NAME(fkc.parent_object_id, fkc.parent_column_id) AS PeopleColumn,
        OBJECT_SCHEMA_NAME(fkc.referenced_object_id) + '.' + 
            OBJECT_NAME(fkc.referenced_object_id) AS ReferencedTable,
        COL_NAME(fkc.referenced_object_id, fkc.referenced_column_id) AS ReferencedColumn
    FROM sys.foreign_keys fk
    JOIN sys.foreign_key_columns fkc ON fk.object_id = fkc.constraint_object_id
    WHERE fk.parent_object_id = OBJECT_ID('Application.People')
    ORDER BY PeopleColumn
""")

C:\Users\blitz\AppData\Local\Temp\ipykernel_17896\243355505.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,PeopleColumn,ReferencedTable,ReferencedColumn
0,LastEditedBy,Application.People,PersonID


Only `LastEditedBy` (system column). **No business FKs to follow — People is self-contained.**
Unlike DimCustomers (8 tables JOINed), DimSalesPeople comes from just 1 table filtered by `IsSalesperson = 1`.

#### 3. Filtered result — only salespeople

In [61]:
# How many salespeople are there?
sql("""
    SELECT PersonID, FullName, PreferredName, LogonName,
           PhoneNumber, FaxNumber, EmailAddress
    FROM Application.People
    WHERE IsSalesperson = 1
""")

C:\Users\blitz\AppData\Local\Temp\ipykernel_17896\243355505.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,PersonID,FullName,PreferredName,LogonName,PhoneNumber,FaxNumber,EmailAddress
0,2,Kayla Woodcock,Kayla,kaylaw@wideworldimporters.com,(415) 555-0102,(415) 555-0103,kaylaw@wideworldimporters.com
1,3,Hudson Onslow,Hudson,hudsono@wideworldimporters.com,(415) 555-0102,(415) 555-0103,hudsono@wideworldimporters.com
2,6,Sophia Hinton,Sophia,sophiah@wideworldimporters.com,(415) 555-0102,(415) 555-0103,sophiah@wideworldimporters.com
3,7,Amy Trefl,Amy,amyt@wideworldimporters.com,(415) 555-0102,(415) 555-0103,amyt@wideworldimporters.com
4,8,Anthony Grosse,Anthony,anthonyg@wideworldimporters.com,(415) 555-0102,(415) 555-0103,anthonyg@wideworldimporters.com
5,13,Hudson Hollinworth,Hudson,hudsonh@wideworldimporters.com,(415) 555-0102,(415) 555-0103,hudsonh@wideworldimporters.com
6,14,Lily Code,Lily,lilyc@wideworldimporters.com,(415) 555-0102,(415) 555-0103,lilyc@wideworldimporters.com
7,15,Taj Shand,Taj,tajs@wideworldimporters.com,(415) 555-0102,(415) 555-0103,tajs@wideworldimporters.com
8,16,Archer Lamble,Archer,archerl@wideworldimporters.com,(415) 555-0102,(415) 555-0103,archerl@wideworldimporters.com
9,20,Jack Potter,Jack,jackp@wideworldimporters.com,(415) 555-0102,(415) 555-0103,jackp@wideworldimporters.com


**Observations:**
- Only 10 salespeople in the entire company
- No FK chains — all data is in this one table, just filtered by `IsSalesperson = 1`
- DimSalesPeople columns: `PersonID` (business key), `FullName`, `PreferredName`, `LogonName`, `PhoneNumber`, `FaxNumber`, `EmailAddress`
- Unlike DimCustomers (8 tables → 1), DimSalesPeople is 1 table → 1 (just filtered)

### DimProducts

OrderLines FK confirmed: `StockItemID` → **Warehouse.StockItems** (StockItemID).
Let's explore.

#### 1. StockItemID → Warehouse.StockItems columns

In [62]:
# What columns does StockItems have?
sql("""
    SELECT COLUMN_NAME, DATA_TYPE, IS_NULLABLE
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = 'Warehouse' AND TABLE_NAME = 'StockItems'
    ORDER BY ORDINAL_POSITION
""")

C:\Users\blitz\AppData\Local\Temp\ipykernel_17896\243355505.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,COLUMN_NAME,DATA_TYPE,IS_NULLABLE
0,StockItemID,int,NO
1,StockItemName,nvarchar,NO
2,SupplierID,int,NO
3,ColorID,int,YES
4,UnitPackageID,int,NO
5,OuterPackageID,int,NO
6,Brand,nvarchar,YES
7,Size,nvarchar,YES
8,LeadTimeDays,int,NO
9,QuantityPerOuter,int,NO


**Observations:**
- `StockItemName`, `Brand`, `Size` — product attributes relevant to the requirement ("brand, colour, and price may impact gross sales")
- `ColorID` — FK, need to follow to get actual color name
- `SupplierID` — FK, interesting — this connects products to suppliers (relevant for the assignment's DimSuppliers)
- `UnitPrice`, `TaxRate` — numeric, but these are product-level defaults. The actual sale prices are in OrderLines.
- Not useful for analysis: `LeadTimeDays`, `Barcode`, `Photo`, `MarketingComments`, `CustomFields` — operational data

**FKs to verify:** `ColorID`, `SupplierID` — where do they point?

#### 2. StockItems FK relationships

In [63]:
# StockItems FK — any chains to follow?
sql("""
    SELECT 
        COL_NAME(fkc.parent_object_id, fkc.parent_column_id) AS StockItemsColumn,
        OBJECT_SCHEMA_NAME(fkc.referenced_object_id) + '.' + 
            OBJECT_NAME(fkc.referenced_object_id) AS ReferencedTable,
        COL_NAME(fkc.referenced_object_id, fkc.referenced_column_id) AS ReferencedColumn
    FROM sys.foreign_keys fk
    JOIN sys.foreign_key_columns fkc ON fk.object_id = fkc.constraint_object_id
    WHERE fk.parent_object_id = OBJECT_ID('Warehouse.StockItems')
    ORDER BY StockItemsColumn
""")

C:\Users\blitz\AppData\Local\Temp\ipykernel_17896\243355505.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,StockItemsColumn,ReferencedTable,ReferencedColumn
0,ColorID,Warehouse.Colors,ColorID
1,LastEditedBy,Application.People,PersonID
2,OuterPackageID,Warehouse.PackageTypes,PackageTypeID
3,SupplierID,Purchasing.Suppliers,SupplierID
4,UnitPackageID,Warehouse.PackageTypes,PackageTypeID


**Confirmed:**
- `ColorID` → **Warehouse.Colors** — need to JOIN for color name
- `SupplierID` → **Purchasing.Suppliers** — this is how products connect to suppliers. Important for the assignment's DimSuppliers!
- `OuterPackageID`, `UnitPackageID` → Warehouse.PackageTypes — not needed for our analysis

Let's see Colors (simple lookup) then the denormalized result.

In [64]:
# ColorID → what's in Colors?
sql("SELECT * FROM Warehouse.Colors")

C:\Users\blitz\AppData\Local\Temp\ipykernel_17896\243355505.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,ColorID,ColorName,LastEditedBy,ValidFrom,ValidTo
0,1,Azure,1,2013-01-01 00:00:00,9999-12-31 23:59:59.999999
1,2,Beige,1,2013-01-01 00:00:00,9999-12-31 23:59:59.999999
2,3,Black,1,2013-01-01 00:00:00,9999-12-31 23:59:59.999999
3,4,Blue,1,2013-01-01 00:00:00,9999-12-31 23:59:59.999999
4,5,Charcoal,1,2013-01-01 00:00:00,9999-12-31 23:59:59.999999
5,6,Chartreuse,1,2013-01-01 00:00:00,9999-12-31 23:59:59.999999
6,7,Cyan,1,2013-01-01 00:00:00,9999-12-31 23:59:59.999999
7,8,Dark Brown,1,2013-01-01 00:00:00,9999-12-31 23:59:59.999999
8,9,Dark Green,1,2013-01-01 00:00:00,9999-12-31 23:59:59.999999
9,10,Fuchsia,1,2013-01-01 00:00:00,9999-12-31 23:59:59.999999


#### 3. Denormalized result — StockItems + Colors

In [65]:
# Products — StockItems JOIN Colors
sql("""
    SELECT TOP 5
        si.StockItemID, si.StockItemName,
        c.ColorName, si.Brand, si.Size
    FROM Warehouse.StockItems si
    LEFT JOIN Warehouse.Colors c ON si.ColorID = c.ColorID
""")

C:\Users\blitz\AppData\Local\Temp\ipykernel_17896\243355505.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,StockItemID,StockItemName,ColorName,Brand,Size
0,1,USB missile launcher (Green),NaN,None,None
1,2,USB rocket launcher (Gray),Steel Gray,None,None
2,3,Office cube periscope (Black),Black,None,None
3,4,USB food flash drive - sushi roll,NaN,None,None
4,5,USB food flash drive - hamburger,NaN,None,None


**Observations:**
- Some products have no `ColorID` (NULL) even when the name contains a color (e.g., "USB missile launcher (Green)" has no ColorID). Source data quality issue — not our problem in Req 1, but Req 5 (Transform) may need to handle this.
- `Brand` and `Size` are also nullable — many products have none.
- These columns must allow NULL in DimProducts.
- DimProducts columns: `StockItemID` (business key), `StockItemName`, `ColorName` (from Colors JOIN), `Brand`, `Size`
- Only 2 tables flattened (StockItems + Colors) — simpler than DimCustomers (8 tables)
- Note: `SupplierID` is in StockItems — links products to suppliers, relevant for the assignment's DimSuppliers

### DimLocation

The requirement says:
> "determine if certain **cities** have brand, colour, and price preferences"
> "selling in certain **cities**"

PDF p.29: "location is a very common dimension so we will have a bias towards defining it as a **dimension** instead of a property."

City is its own analysis axis, not just a customer attribute. That's why DimLocation exists separately.

FK chain confirmed in DimCustomers above:
```
Customers.DeliveryCityID → Application.Cities (CityName)
  → Application.StateProvinces (StateProvinceCode, StateProvinceName)
    → Application.Countries (CountryName, FormalName) — end of chain
```

#### 1. Source tables — already explored in DimCustomers

We already verified the full FK chain and columns:
- **Application.Cities**: CityName, StateProvinceID → confirmed FK to StateProvinces
- **Application.StateProvinces**: StateProvinceCode, StateProvinceName, CountryID → confirmed FK to Countries
- **Application.Countries**: CountryName, FormalName → end of chain

DimLocation = these 3 tables flattened into 1. Let's see the result.

#### 2. Denormalized result

In [66]:
# Location chain: Cities → StateProvinces → Countries (3 tables JOINed into 1 Dim)
sql("""
    SELECT TOP 5
        c.CityID, c.CityName,
        sp.StateProvinceCode, sp.StateProvinceName,
        co.CountryName, co.FormalName
    FROM Application.Cities c
    JOIN Application.StateProvinces sp ON c.StateProvinceID = sp.StateProvinceID
    JOIN Application.Countries co ON sp.CountryID = co.CountryID
""")

C:\Users\blitz\AppData\Local\Temp\ipykernel_17896\243355505.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,CityID,CityName,StateProvinceCode,StateProvinceName,CountryName,FormalName
0,5,Abbeville,GA,Georgia,United States,United States of America
1,75,Acree,GA,Georgia,United States,United States of America
2,82,Acworth,GA,Georgia,United States,United States of America
3,94,Adairsville,GA,Georgia,United States,United States of America
4,139,Adel,GA,Georgia,United States,United States of America


**Observations:**
- 3 tables (Cities + StateProvinces + Countries) → 1 flat row per city
- DimLocation columns: `CityName`, `StateProvinceCode`, `StateProvinceName`, `CountryName`, `CountryFormalName`
- `CityID` = business key for this dimension
- Each city appears once — unlike DimCustomers where the same city repeats for every customer in that city

### DimSuppliers (assignment addition)

Not in the class's original 5 Dims. Discovered via StockItems FK: `SupplierID → Purchasing.Suppliers`.

The assignment says:
> "determine whether certain **suppliers** products show more successful sales"
> "add the **SupplierCategoryName** field to the appropriate table in the dimensional model"

#### 1. SupplierID → Purchasing.Suppliers columns

In [68]:
# What columns does Suppliers have?
sql("""
    SELECT COLUMN_NAME, DATA_TYPE, IS_NULLABLE
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = 'Purchasing' AND TABLE_NAME = 'Suppliers'
    ORDER BY ORDINAL_POSITION
""")

C:\Users\blitz\AppData\Local\Temp\ipykernel_17896\243355505.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,COLUMN_NAME,DATA_TYPE,IS_NULLABLE
0,SupplierID,int,NO
1,SupplierName,nvarchar,NO
2,SupplierCategoryID,int,NO
3,PrimaryContactPersonID,int,NO
4,AlternateContactPersonID,int,NO
5,DeliveryMethodID,int,YES
6,DeliveryCityID,int,NO
7,PostalCityID,int,NO
8,SupplierReference,nvarchar,YES
9,BankAccountName,nvarchar,YES


**Observations:**
- Structure is very similar to Sales.Customers — same pattern of IDs + addresses + bank info
- `SupplierName`, `PhoneNumber`, `FaxNumber`, `WebsiteURL` — useful for DimSuppliers
- `SupplierCategoryID` — FK, need to follow (assignment explicitly says to add SupplierCategoryName)
- `DeliveryCityID`, `PostalCityID` — same Cities chain as Customers, but not needed for DimSuppliers (location analysis is via DimLocation)
- Not useful: `BankAccount*`, `PaymentDays`, `DeliveryAddress*` — operational/financial data

**FKs to verify:** `SupplierCategoryID` — where does it point?

#### 2. Suppliers FK relationships

In [69]:
# Suppliers FK — any chains to follow?
sql("""
    SELECT 
        COL_NAME(fkc.parent_object_id, fkc.parent_column_id) AS SuppliersColumn,
        OBJECT_SCHEMA_NAME(fkc.referenced_object_id) + '.' + 
            OBJECT_NAME(fkc.referenced_object_id) AS ReferencedTable,
        COL_NAME(fkc.referenced_object_id, fkc.referenced_column_id) AS ReferencedColumn
    FROM sys.foreign_keys fk
    JOIN sys.foreign_key_columns fkc ON fk.object_id = fkc.constraint_object_id
    WHERE fk.parent_object_id = OBJECT_ID('Purchasing.Suppliers')
    ORDER BY SuppliersColumn
""")

C:\Users\blitz\AppData\Local\Temp\ipykernel_17896\243355505.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,SuppliersColumn,ReferencedTable,ReferencedColumn
0,AlternateContactPersonID,Application.People,PersonID
1,DeliveryCityID,Application.Cities,CityID
2,DeliveryMethodID,Application.DeliveryMethods,DeliveryMethodID
3,LastEditedBy,Application.People,PersonID
4,PostalCityID,Application.Cities,CityID
5,PrimaryContactPersonID,Application.People,PersonID
6,SupplierCategoryID,Purchasing.SupplierCategories,SupplierCategoryID


**Confirmed:**
- `SupplierCategoryID` → **Purchasing.SupplierCategories** — this is the one the assignment asks us to include
- `DeliveryCityID`, `PostalCityID` → Application.Cities — same chain as Customers, but not needed for DimSuppliers
- `PrimaryContactPersonID`, `AlternateContactPersonID` → Application.People — not needed

Only `SupplierCategoryID` matters for DimSuppliers. Simple JOIN like CustomerCategories was.

#### 3. Denormalized result

In [70]:
# Suppliers + SupplierCategories JOIN (assumption — verify after FK results)
sql("""
    SELECT TOP 5
        s.SupplierID, s.SupplierName,
        s.PhoneNumber, s.FaxNumber, s.WebsiteURL,
        sc.SupplierCategoryName
    FROM Purchasing.Suppliers s
    JOIN Purchasing.SupplierCategories sc ON s.SupplierCategoryID = sc.SupplierCategoryID
""")

C:\Users\blitz\AppData\Local\Temp\ipykernel_17896\243355505.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,SupplierID,SupplierName,PhoneNumber,FaxNumber,WebsiteURL,SupplierCategoryName
0,1,A Datum Corporation,(847) 555-0100,(847) 555-0101,http://www.adatum.com,Novelty Goods Supplier
1,2,"Contoso, Ltd.",(360) 555-0100,(360) 555-0101,http://www.contoso.com,Novelty Goods Supplier
2,3,Consolidated Messenger,(415) 555-0100,(415) 555-0101,http://www.consolidatedmessenger.com,Courier Services Supplier
3,4,"Fabrikam, Inc.",(203) 555-0104,(203) 555-0108,http://www.fabrikam.com,Clothing Supplier
4,5,Graphic Design Institute,(406) 555-0105,(406) 555-0106,http://www.graphicdesigninstitute.com,Novelty Goods Supplier


**Observations:**
- 2 tables flattened (Suppliers + SupplierCategories) — same pattern as DimProducts (StockItems + Colors)
- `SupplierID` = business key
- DimSuppliers columns: `SupplierName`, `PhoneNumber`, `FaxNumber`, `WebsiteURL`, `SupplierCategoryName`
- Assignment requires **SCD Type 2** — "if non-key attributes change, they would impact associated facts"
- SCD Type 2 means: if a supplier changes name or category, keep both old and new records with EffectiveDate/EndDate/IsCurrent

### DimDate

Orders has `OrderDate` — just a date value like `2013-01-01`. But that alone can't answer:
- "Which **month** had the most sales?" → need Month
- "**Q1 vs Q2** revenue comparison?" → need Quarter
- "Do we sell more on **Tuesdays**?" → need DayOfWeekName

You could calculate `MONTH(OrderDate)` every time, but that's slow on 231K rows.

DimDate pre-calculates all of this:
```
DateKey     DateValue    CYear  CMonth  CQtr  DayOfWeekName  MonthName
20130101    2013-01-01   2013   1       1     Tuesday        January
20130102    2013-01-02   2013   1       1     Wednesday      January
```

FactSales references `DateKey = 20130101` → JOIN to get Year, Month, Quarter, DayOfWeek instantly.
This is denormalization — the values are calculable, but pre-stored for read performance.

Unlike other Dims, DimDate has **no source table** in WideWorldImporters — it's generated by a stored procedure.
The class created `DimDate_Load` for this — covered in **Req 2**. For Req 1, we just need the table structure.

### Step 3 Summary: 3NF → Star Schema

**BEFORE (3NF) — what WideWorldImporters looks like now:**
```
Sales.Orders ──→ Sales.Customers ──→ Sales.CustomerCategories
    │                  │
    │                  ├──→ Application.Cities ──→ Application.StateProvinces ──→ Application.Countries
    │                  │
    │                  └──→ Application.Cities (Postal) ──→ ...
    │
    ├──→ Application.People
    │
    └──→ Sales.OrderLines ──→ Warehouse.StockItems ──→ Warehouse.Colors
                                       │
                                       └──→ Purchasing.Suppliers ──→ Purchasing.SupplierCategories
```
13+ tables, deeply nested FK chains. Every analytical query needs multiple JOINs.

**AFTER (Star Schema) — what WWI_DM will look like:**
```
    DimCustomers                                                          DimProducts
    - CustomerName                                                        - ProductName
    - CategoryName                                                        - ColorName
          │                                                               - Brand, Size
      CustomerKey                                                              │
          │                                                               ProductKey
          │               DimDate                                              │
          │            - Year, Month                                           │
          │            - Quarter, DayOfWeek                                    │
          │                  │                                                 │
          │               DateKey                                              │
          │                  │                                                 │
          └──────────────────┼──────────── FactSales ──────────────────────────┘
                             │            - Quantity
          ┌──────────────────┼────────────- UnitPrice───────────────────────────┐
          │                  │            - TaxRate                             │
          │                  │            - TotalBeforeTax                      │
     LocationKey             │            - TotalAfterTax              SalespersonKey
          │            SupplierKey                                             │
    DimLocation              │                                          DimSalesPeople
    - CityName         DimSuppliers                                     - FullName
    - StateProv        - SupplierName                                   - Phone, Email
    - Country          - CategoryName
                       - Phone, Fax, URL
```
6 Dims, each 1 flat table. Any question = 1 JOIN to the relevant Dim.

**Every Dim collapses a 3NF chain into 1 flat table:**

| Dim | Source tables (3NF) | Flattened |
|-----|---------------------|-----------|
| DimCustomers | Customers + CustomerCategories (+ Cities chain in class design) | 2~8 → 1 |
| DimSalesPeople | People (filtered by IsSalesperson=1) | 1 → 1 |
| DimProducts | StockItems + Colors | 2 → 1 |
| DimLocation | Cities + StateProvinces + Countries | 3 → 1 |
| DimSuppliers | Suppliers + SupplierCategories | 2 → 1 |
| DimDate | None (calculated from dates) | Generated |

FactSales (center) holds measures: Quantity, UnitPrice, TaxRate, TotalBeforeTax, TotalAfterTax

## Step 4 — Define Facts and Measures

In Step 1, we saw OrderLines has numeric columns (`Quantity`, `UnitPrice`, `TaxRate`) and marked them as "verify later."
Now let's confirm: which of these become measures in FactSales?

The requirement says:
> "determine how things like brand, colour, and **price** may impact **gross sales**"

"Gross sales" = total revenue. So we need price × quantity type calculations.

![p.33](images/week7-p33.png)
![p.34](images/week7-p34.png)
![p.35](images/week7-p35.png)

In [71]:
# What numeric columns does OrderLines have?
sql("""
    SELECT COLUMN_NAME, DATA_TYPE, NUMERIC_PRECISION, NUMERIC_SCALE
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = 'Sales' AND TABLE_NAME = 'OrderLines'
      AND DATA_TYPE IN ('int','decimal','numeric','money','float','bigint')
    ORDER BY ORDINAL_POSITION
""")

C:\Users\blitz\AppData\Local\Temp\ipykernel_17896\243355505.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,COLUMN_NAME,DATA_TYPE,NUMERIC_PRECISION,NUMERIC_SCALE
0,OrderLineID,int,10,0
1,OrderID,int,10,0
2,StockItemID,int,10,0
3,PackageTypeID,int,10,0
4,Quantity,int,10,0
5,UnitPrice,decimal,18,2
6,TaxRate,decimal,18,3
7,PickedQuantity,int,10,0
8,LastEditedBy,int,10,0


Not every numeric column is a measure. A measure must be **analytically meaningful when aggregated** (SUM, AVG):
- `Quantity`, `UnitPrice`, `TaxRate` → SUM(Quantity) = total units sold. AVG(UnitPrice) = average price. ✅
- `OrderLineID`, `OrderID`, `StockItemID` → SUM(OrderID) = meaningless. These are IDs/FKs, not measures. ❌
- `PickedQuantity` → warehouse operational data, not sales analysis. ❌

In [72]:
# See actual values — which of these make good measures?
sql("""
    SELECT TOP 5
        Quantity, UnitPrice, TaxRate,
        Quantity * UnitPrice AS TotalBeforeTax,
        Quantity * UnitPrice * (1 + TaxRate/100) AS TotalAfterTax
    FROM Sales.OrderLines
""")

C:\Users\blitz\AppData\Local\Temp\ipykernel_17896\243355505.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,Quantity,UnitPrice,TaxRate,TotalBeforeTax,TotalAfterTax
0,50,112.0,15.0,5600.0,6440.00
1,10,230.0,15.0,2300.0,2645.00
2,9,13.0,15.0,117.0,134.55
3,72,18.0,15.0,1296.0,1490.40
4,90,32.0,15.0,2880.0,3312.00


**Observations:**
- `Quantity` — how many units were ordered. Aggregatable (SUM, AVG). ✅ measure
- `UnitPrice` — price per unit. ✅ measure
- `TaxRate` — tax percentage. ✅ measure
- `TotalBeforeTax` = Quantity × UnitPrice — calculated, doesn't exist in OrderLines as a column
- `TotalAfterTax` = Quantity × UnitPrice × (1 + TaxRate/100) — also calculated

**Why store calculated values?** This is denormalization again.
You could calculate them every time with `SUM(Quantity * UnitPrice)`, but:
- Pre-calculating is faster on 231K rows
- In DW, read performance > storage efficiency
- PDF p.35: "In a normalized ER data model, these would violate the third normal form. In a dimensional model, it is acceptable."

**FactSales measures (final):** Quantity, UnitPrice, TaxRate, TotalBeforeTax, TotalAfterTax

## Step 5 — Result

See Step 3 Summary above — 3NF → Star Schema diagram, 6 Dims + FactSales mapping table already covered there.

**Notes:** *(write what you learned here)*

-